In [1]:
import os
from pyspark.sql import SparkSession
import pandas as pd

# Set JAVA_HOME for PySpark
os.environ['JAVA_HOME'] = '/opt/homebrew/Cellar/openjdk@21/21.0.9/libexec/openjdk.jdk/Contents/Home'

# Stop any existing spark sessions
SparkSession._instantiatedSession = None if SparkSession._instantiatedSession else None

In [2]:
from pyspark.sql.functions import current_timestamp, regexp_replace, col, when, coalesce, lit, length

In [3]:
spark = SparkSession.builder \
    .appName("LendingClub") \
    .master("local[2]") \
    .enableHiveSupport() \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/04 10:43:03 WARN Utils: Your hostname, Himanshus-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.215 instead (on interface en0)
26/02/04 10:43:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/04 10:43:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Customers

In [ ]:
file_path = "../data/raw/customers.csv"

In [ ]:
customers_df = spark.read\
        .format("csv")\
        .option("header", "true")\
        .option("inferSchema", "true")\
        .load(file_path)

In [ ]:
customers_df.show(5)

In [ ]:
customers_df_renamed = customers_df.withColumnRenamed("annual_inc", "annual_income")\
                            .withColumnRenamed("addr_state", "address_state")\
                            .withColumnRenamed("zip_code", "address_zipcode")\
                            .withColumnRenamed("country", "address_country")\
                            .withColumnRenamed("tot_hi_cred_lim", "total_high_credit_limit")\
                            .withColumnRenamed("annual_inc_joint", "join_annual_income")

In [ ]:
customers_df_ingested = customers_df_renamed.withColumn("ingest_date", current_timestamp())

In [ ]:
customers_df_ingested.printSchema()

In [ ]:
customers_df_distinct = customers_df_ingested.distinct()

In [ ]:
customers_df_distinct.createOrReplaceTempView("customers")

In [ ]:
customers_income_filtered = spark.sql("select * from customers where annual_income is not null")

In [ ]:
customers_income_filtered.show(5)

In [ ]:
customers_income_filtered.createOrReplaceTempView("customers")

In [ ]:
spark.sql("select distinct(emp_length) from customers").show()

In [ ]:
spark.sql("SELECT regexp_replace(emp_length, '[^0-9]', '') AS emp_length FROM customers").show()

In [ ]:
customers_emplength_cleaned = customers_income_filtered.withColumn("emp_length", regexp_replace(col("emp_length"), "[^0-9]",""))

In [ ]:
customers_emplength_cleaned.show(5)

In [ ]:
customers_emplength_cleaned.printSchema()

In [ ]:
#Replace empty strings with NULL before casting
customers_emplength_int = customers_emplength_cleaned.withColumn(
    "emp_length",
    when(col("emp_length") == "", None).otherwise(col("emp_length")).cast('int')
)

In [ ]:
customers_emplength_int.createOrReplaceTempView("customers")

In [ ]:
avg_emp_length = spark.sql("select floor(avg(emp_length)) as avg_emp_length from customers").collect()
print(avg_emp_length)

In [ ]:
avg_emp_duration = avg_emp_length[0][0]
print(avg_emp_duration)

In [ ]:
customers_emplength_replaced = customers_emplength_int.na.fill(avg_emp_duration, subset=['emp_length'])

In [ ]:
customers_emplength_replaced.createOrReplaceTempView("customers")
spark.sql("select distinct(address_state) from customers").show()

In [ ]:
spark.sql("select count(address_state) from customers where length(address_state)>2").show()

In [ ]:
customers_state_cleaned = customers_emplength_replaced.withColumn(
    "address_state",
    when(length(col("address_state"))> 2, "NA").otherwise(col("address_state"))
)

In [ ]:
customers_state_cleaned.show()

In [ ]:
customers_state_cleaned.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", "../data/cleaned/customers.csv") \
.save()

Loans

In [ ]:
loans_schema = 'loan_id string, member_id string, loan_amount float, funded_amount float, loan_term_months string, interest_rate float, monthly_installment float, issue_date string, loan_status string, loan_purpose string, loan_title string'

In [ ]:
loans_raw_df = spark.read \
.format("csv") \
.option("header",True) \
.schema(loans_schema) \
.load("../data/raw/loans.csv")

loans_raw_df.show(5)

In [ ]:
loans_raw_df.printSchema()

In [ ]:
loans_df_ingestd = loans_raw_df.withColumn("ingest_date", current_timestamp())

In [ ]:
loans_df_ingestd.createOrReplaceTempView("loans")

In [ ]:
spark.sql("select * from loans where loan_amount is null").show(5)


In [ ]:
columns_with_na = ["loan_amount", "funded_amount", "loan_term_months", "interest_rate", "monthly_installment", "issue_date", "loan_status", "loan_purpose", "loan_title"]

In [ ]:
loans_filtered_df = loans_df_ingestd.na.drop(subset = columns_with_na)

In [ ]:
loans_filtered_df.createOrReplaceTempView("loans")

In [ ]:
loans_term_modified_df = loans_filtered_df.withColumn("loan_term_months", 
                                                      (regexp_replace(col("loan_term_months"), " months", "") \
.cast("int") / 12).cast("int")).withColumnRenamed("loan_term_months","loan_term_years")

In [ ]:
loans_term_modified_df.show(5)
loans_term_modified_df.printSchema()

In [ ]:
loans_term_modified_df.createOrReplaceTempView("loans")

In [ ]:
spark.sql("select distinct(loan_purpose) from loans").show()

In [ ]:
spark.sql("select loan_purpose, count(*) as total from loans group by loan_purpose order by total desc").show()

In [ ]:
loan_purposes_valid = ["debt_consolidation", "credit_card", "home_improvement", "other", "major_purchase", "medical", "small_business", "car", "vacation", "moving", "house", "wedding", "renewable_energy", "educational"]

In [ ]:
loans_purpose_modified = loans_term_modified_df.withColumn("loan_purpose", when(col("loan_purpose").isin(loan_purposes_valid), col("loan_purpose")).otherwise("other"))

In [ ]:
loans_purpose_modified.createOrReplaceTempView("loans")

In [ ]:
spark.sql("select loan_purpose, count(*) as total from loans group by loan_purpose order by total desc").show()

In [ ]:
loans_purpose_modified.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", "../data/cleaned/loans.csv")\
.save()

Loan repayments

In [ ]:
loans_repay_raw_df = spark.read \
.format("csv") \
.option("header",True) \
.option("inferSchema", True) \
.load("../data/raw/repayments.csv")

loans_repay_raw_df.show()

In [ ]:
loans_repay_raw_df.printSchema()

In [ ]:
loans_repay_schema = 'loan_id string, total_principal_received float, total_interest_received float, total_late_fee_received float, total_payment_received float, last_payment_amount float, last_payment_date string, next_payment_date string'

In [ ]:
loans_repay_raw_df = spark.read \
.format("csv") \
.option("header",True) \
.schema(loans_repay_schema) \
.load("../data/raw/repayments.csv")

loans_repay_raw_df.printSchema()

In [ ]:
loans_repay_df_ingestd = loans_repay_raw_df.withColumn("ingest_date", current_timestamp())

loans_repay_df_ingestd.printSchema()

In [ ]:
loans_repay_df_ingestd.createOrReplaceTempView("loan_repayments")

In [ ]:
spark.sql("select count(*) from loan_repayments where total_principal_received is null").show()

In [ ]:
columns_to_check = ["total_principal_received", "total_interest_received", "total_late_fee_received", "total_payment_received"]

In [ ]:
loans_repay_filtered_df = loans_repay_df_ingestd.na.drop(subset=columns_to_check)

In [ ]:
loans_repay_filtered_df.createOrReplaceTempView("loan_repayments")

In [ ]:
spark.sql("select count(*) from loan_repayments where total_payment_received = 0.0").show()

In [ ]:
spark.sql("select count(*) from loan_repayments where total_payment_received = 0.0 and total_principal_received != 0.0").show()

In [ ]:
loans_payments_fixed_df = loans_repay_filtered_df.withColumn(
   "total_payment_received",
    when(
        (col("total_principal_received") != 0.0) &
        (col("total_payment_received") == 0.0),
        col("total_principal_received") + col("total_interest_received") + col("total_late_fee_received")
    ).otherwise(col("total_payment_received"))
)

In [ ]:
loans_payments_fixed_df.show()

In [ ]:
loans_payments_fixed2_df = loans_payments_fixed_df.filter("total_payment_received != 0.0")

In [ ]:
loans_payments_fixed2_df.createOrReplaceTempView("loan_repayments")

In [ ]:
spark.sql("select last_payment_date, count(*) as total from loan_repayments group by last_payment_date order by total").show(178)

In [ ]:
loans_payments_ldate_fixed_df = loans_payments_fixed2_df.withColumn(
    "next_payment_date",
    when(col("last_payment_date").rlike(r'^\d+(\.\d+)?$'), None)\
        .otherwise(col("last_payment_date"))
)

In [ ]:
loans_payments_ndate_fixed_df = loans_payments_ldate_fixed_df.withColumn(
  "last_payment_date",
   when(
       (col("next_payment_date").rlike(r'^\d+(\.\d+)?$')),
       None
       ).otherwise(col("next_payment_date"))
)

In [ ]:
loans_payments_ndate_fixed_df.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", "../data/cleaned/repayments.csv")\
.save()

Cleaning delinquencies

In [ ]:
loans_def_raw_df = spark.read \
.format("csv") \
.option("header",True) \
.option("inferSchema", True) \
.load("../data/raw/delinquencies.csv")

In [ ]:
loans_def_raw_df.show()

In [ ]:
loans_def_raw_df.printSchema()

In [ ]:
loans_def_raw_df.createOrReplaceTempView("loan_defaulters")

In [ ]:
spark.sql("select delinq_2yrs, count(*) as total from loan_defaulters group by delinq_2yrs order by total desc").show()

In [ ]:
loan_defaulters_schema = "member_id string, delinq_2yrs float, delinq_amnt float, pub_rec float, pub_rec_bankruptcies float,inq_last_6mths float, total_rec_late_fee float, mths_since_last_delinq float, mths_since_last_record float"

In [ ]:
loans_def_raw_df = spark.read \
.format("csv") \
.option("header",True) \
.schema(loan_defaulters_schema) \
.load("../data/raw/delinquencies.csv")

In [ ]:
loans_def_raw_df.createOrReplaceTempView("loan_defaulters")

In [ ]:
loans_def_processed_df = loans_def_raw_df.withColumn("delinq_2yrs", col("delinq_2yrs").cast("integer")).fillna(0, subset = ["delinq_2yrs"])

In [ ]:
loans_def_processed_df.createOrReplaceTempView("loan_defaulters")

In [ ]:
spark.sql("select count(*) from loan_defaulters where delinq_2yrs is null").show()

In [ ]:
loans_def_delinq_df = spark.sql("select member_id,delinq_2yrs, delinq_amnt, int(mths_since_last_delinq) from loan_defaulters where delinq_2yrs > 0 or mths_since_last_delinq > 0")

In [ ]:
loans_def_delinq_df.show()

In [ ]:
loans_def_records_enq_df = spark.sql("select member_id from loan_defaulters where pub_rec > 0.0 or pub_rec_bankruptcies > 0.0 or inq_last_6mths > 0.0")

In [ ]:
loans_def_records_enq_df.show()

In [ ]:
loans_def_delinq_df.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", "../data/cleaned/delinquencies.csv") \
.save()

In [ ]:
loans_def_records_enq_df.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", "../data/cleaned/delinquencies_public_records.csv") \
.save()

Creating Defaulters details

In [ ]:
loans_def_p_pub_rec_df = loans_def_processed_df.withColumn("pub_rec", col("pub_rec").cast("integer")).fillna(0, subset = ["pub_rec"])

In [ ]:
loans_def_p_pub_rec_bankruptcies_df = loans_def_p_pub_rec_df.withColumn("pub_rec_bankruptcies", col("pub_rec_bankruptcies").cast("integer")).fillna(0, subset = ["pub_rec_bankruptcies"])

In [ ]:
loans_def_p_inq_last_6mths_df = loans_def_p_pub_rec_bankruptcies_df.withColumn("inq_last_6mths", col("inq_last_6mths").cast("integer")).fillna(0, subset = ["inq_last_6mths"])

In [ ]:
loans_def_p_inq_last_6mths_df.createOrReplaceTempView("loan_defaulters")

In [ ]:
loans_def_detail_records_enq_df = spark.sql("select member_id, pub_rec, pub_rec_bankruptcies, inq_last_6mths from loan_defaulters")

In [ ]:
loans_def_detail_records_enq_df.show()

In [ ]:
loans_def_detail_records_enq_df.write \
.option("header", True) \
.format("csv") \
.mode("overwrite") \
.option("path", "../data/cleaned/loans_del_detail_records_enq_df.csv") \
.save()

Create permanent tables on the datasets

In [ ]:
spark.sql("create database lending_club")

In [ ]:
spark.sql("drop table if exists lending_club.loans")

In [ ]:
spark.sql("""create external table lending_club.customers(
member_id string, emp_title string, emp_length int, 
home_ownership string, annual_income float, address_state string, address_zipcode string, address_country string, grade string, 
sub_grade string, verification_status string, total_high_credit_limit float, application_type string, join_annual_income float, 
verification_status_joint string, ingest_date timestamp)
using csv location  '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned/customers.csv'
options('header'='true', 'inferSchema'='false')
""")

In [ ]:
spark.sql("select * from lending_club.customers").show()

In [ ]:
spark.sql("""
create external table lending_club.loans(
loan_id string, member_id string, loan_amount float, funded_amount float,
loan_term_years integer, interest_rate float, monthly_installment float, issue_date string,
loan_status string, loan_purpose string, loan_title string, ingest_date timestamp)
using csv location  '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned/loans.csv'
options('header'='true', 'inferSchema'='false')
""")

In [ ]:
spark.sql("select * from lending_club.loans").show()

In [ ]:
spark.sql("""CREATE EXTERNAL TABLE lending_club.loans_repayments(
    loan_id string, total_principal_received float, 
    total_interest_received float,total_late_fee_received float,
    total_payment_received float,last_payment_amount float,
    last_payment_date string,next_payment_date string,
    ingest_date timestamp)
    using csv location  '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned/repayments.csv'
    options('header'='true', 'inferSchema'='false')
""")

In [ ]:
spark.sql("select * from lending_club.loans_repayments").show()

In [ ]:
spark.sql("""CREATE EXTERNAL TABLE lending_club.loans_defaulters_delinq(
    member_id string, delinq_2yrs integer, delinq_amnt float,
    mths_since_last_delinq integer)
    using csv location  '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned/delinquencies.csv'
    options('header'='true', 'inferSchema'='false')
""")

In [ ]:
spark.sql("select * from lending_club.loans_defaulters_delinq").show()

In [ ]:
spark.sql("""CREATE EXTERNAL TABLE lending_club.loans_defaulters_detail_rec_enq(
    member_id string, pub_rec integer, pub_rec_bankruptcies integer, 
    inq_last_6mths integer)
    using csv location  '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned/loans_del_detail_records_enq_df.csv'
    options('header'='true', 'inferSchema'='false')
    """)

In [ ]:
spark.sql("select * from lending_club.loans_defaulters_detail_rec_enq").show()

In [ ]:
spark.sql("""
create or replace view lending_club.customers_loan_v as select
l.loan_id,
c.member_id,
c.emp_title,
c.emp_length,
c.home_ownership,
c.annual_income,
c.address_state,
c.address_zipcode,
c.address_country,
c.grade,
c.sub_grade,
c.verification_status,
c.total_high_credit_limit,
c.application_type,
c.join_annual_income,
c.verification_status_joint,
l.loan_amount,
l.funded_amount,
l.loan_term_years,
l.interest_rate,
l.monthly_installment,
l.issue_date,
l.loan_status,
l.loan_purpose,
r.total_principal_received,
r.total_interest_received,
r.total_late_fee_received,
r.last_payment_date,
r.next_payment_date,
d.delinq_2yrs,
d.delinq_amnt,
d.mths_since_last_delinq,
e.pub_rec,
e.pub_rec_bankruptcies,
e.inq_last_6mths

FROM lending_club.customers c
LEFT JOIN lending_club.loans l on c.member_id = l.member_id
LEFT JOIN lending_club.loans_repayments r ON l.loan_id = r.loan_id
LEFT JOIN lending_club.loans_defaulters_delinq d ON c.member_id = d.member_id
LEFT JOIN lending_club.loans_defaulters_detail_rec_enq e ON c.member_id = e.member_id
""")

In [ ]:
spark.sql("select * from lending_club.customers_loan_v").show()

In [ ]:
spark.sql("""
create table lending_club.customers_loan_t as select
l.loan_id,
c.member_id,
c.emp_title,
c.emp_length,
c.home_ownership,
c.annual_income,
c.address_state,
c.address_zipcode,
c.address_country,
c.grade,
c.sub_grade,
c.verification_status,
c.total_high_credit_limit,
c.application_type,
c.join_annual_income,
c.verification_status_joint,
l.loan_amount,
l.funded_amount,
l.loan_term_years,
l.interest_rate,
l.monthly_installment,
l.issue_date,
l.loan_status,
l.loan_purpose,
r.total_principal_received,
r.total_interest_received,
r.total_late_fee_received,
r.last_payment_date,
r.next_payment_date,
d.delinq_2yrs,
d.delinq_amnt,
d.mths_since_last_delinq,
e.pub_rec,
e.pub_rec_bankruptcies,
e.inq_last_6mths

FROM lending_club.customers c
LEFT JOIN lending_club.loans l on c.member_id = l.member_id
LEFT JOIN lending_club.loans_repayments r ON l.loan_id = r.loan_id
LEFT JOIN lending_club.loans_defaulters_delinq d ON c.member_id = d.member_id
LEFT JOIN lending_club.loans_defaulters_detail_rec_enq e ON c.member_id = e.member_id
""")

In [ ]:
spark.sql("select * from lending_club.customers_loan_t").show()   

Remove duplicate member ids

In [ ]:
spark.sql("""select member_id, count(*) as total 
from lending_club.customers
group by member_id order by total desc
""").show()

In [ ]:
spark.sql("""select member_id, count(*) as total 
from lending_club.loans_defaulters_delinq
group by member_id order by total desc
""").show()

In [ ]:
spark.sql("""select member_id, count(*) as total 
from lending_club.loans_defaulters_detail_rec_enq
group by member_id order by total desc
""").show()

In [ ]:
bad_data_customer_df = spark.sql("""select member_id from(select member_id, count(*)
as total from lending_club.customers
group by member_id having total > 1)""")

In [ ]:
bad_data_customer_df.show()

In [ ]:
bad_data_customer_df.count()

In [ ]:
bad_data_loans_defaulters_delinq_df = spark.sql("""select member_id from(select member_id, count(*)
as total from lending_club.loans_defaulters_delinq
group by member_id having total > 1)""")

In [ ]:
bad_data_loans_defaulters_delinq_df.count()

In [ ]:
bad_data_loans_defaulters_detail_rec_enq_df = spark.sql("""select member_id from(select member_id, count(*)
as total from lending_club.loans_defaulters_detail_rec_enq
group by member_id having total > 1)""")

bad_data_loans_defaulters_detail_rec_enq_df.count()

In [ ]:
bad_data_customer_df.repartition(1).write \
.format("csv") \
.option("header", True) \
.mode("overwrite") \
.option("path", "../data/bad/bad_data_customers.csv") \
.save()

In [ ]:
bad_data_loans_defaulters_delinq_df.repartition(1).write \
.format("csv") \
.option("header", True) \
.mode("overwrite") \
.option("path", "../data/bad/bad_data_loans_defaulters_delinq.csv") \
.save()

In [ ]:
bad_customer_data_df = bad_data_customer_df.select("member_id") \
.union(bad_data_loans_defaulters_delinq_df.select("member_id")) \
.union(bad_data_loans_defaulters_detail_rec_enq_df.select("member_id"))

In [ ]:
bad_customer_data_final_df = bad_customer_data_df.distinct()


In [ ]:
bad_customer_data_final_df.count()
bad_customer_data_final_df.show()

In [ ]:
bad_customer_data_final_df.repartition(1).write \
.format("csv") \
.option("header", True) \
.mode("overwrite") \
.option("path", "../data/bad/bad_customer_data_final.csv") \
.save()

In [ ]:
bad_customer_data_final_df.createOrReplaceTempView("bad_data_customer")

In [ ]:
customers_df = spark.sql("""select * from lending_club.customers
where member_id NOT IN (select member_id from bad_data_customer)
""")


In [ ]:
customers_df.show()

In [ ]:
customers_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "../data/cleaned_new/customers_parquet") \
.save()

In [ ]:
temp = spark.read.parquet("../data/cleaned_new/customers_parquet")
temp.show()

In [ ]:
loans_defaulters_delinq_df = spark.sql("""select * from lending_club.loans_defaulters_delinq
where member_id NOT IN (select member_id from bad_data_customer)
""")

In [ ]:
loans_defaulters_delinq_df.show()

In [ ]:
loans_defaulters_delinq_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "../data/cleaned_new/loans_defaulters_delinq_parquet") \
.save()

In [ ]:
temp = spark.read.parquet("../data/cleaned_new/loans_defaulters_delinq_parquet")
temp.show()

In [ ]:
loans_defaulters_detail_rec_enq_df = spark.sql("""select * from lending_club.loans_defaulters_detail_rec_enq
where member_id NOT IN (select member_id from bad_data_customer)
""")

In [ ]:
loans_defaulters_detail_rec_enq_df.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "../data/cleaned_new/loans_defaulters_detail_rec_enq_parquet") \
.save()

In [ ]:
temp = spark.read.parquet("../data/cleaned_new/loans_defaulters_detail_rec_enq_parquet")
temp.show()

In [ ]:
spark.sql("DROP TABLE IF EXISTS lending_club.loans_defaulters_detail_rec_enq_new")

In [ ]:
spark.sql("""
create EXTERNAL TABLE lending_club.customers_new(
    member_id string, emp_title string, emp_length int, home_ownership string, 
    annual_income float, address_state string, address_zipcode string, 
    address_country string, grade string, sub_grade string, verification_status string,
    total_high_credit_limit float, application_type string, 
    join_annual_income float, verification_status_joint string, ingest_date timestamp)
    stored as parquet
    LOCATION '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned_new/customers_parquet'
    
""")

In [ ]:
spark.sql("select * from lending_club.customers_new").show()

In [ ]:
spark.sql("""
create EXTERNAL TABLE lending_club.loans_defaulters_delinq_new(
    member_id string,delinq_2yrs integer, delinq_amnt float, mths_since_last_delinq integer)
    stored as parquet
    LOCATION '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned_new/loans_defaulters_delinq_parquet'
""")

In [ ]:
spark.sql("select * from lending_club.loans_defaulters_delinq_new").show()

In [ ]:
spark.sql("""
create EXTERNAL TABLE lending_club.loans_defaulters_detail_rec_enq_new(
    member_id string, pub_rec integer, pub_rec_bankruptcies integer, inq_last_6mths integer)
    stored as parquet
    LOCATION '/Users/himanshuhegde/Desktop/Lending_club/data/cleaned_new/loans_defaulters_detail_rec_enq_parquet'
""")

In [ ]:
spark.sql("select * from lending_club.loans_defaulters_detail_rec_enq_new").show()

In [ ]:
spark.sql("""select member_id, count(*) as total 
from lending_club.customers_new
group by member_id order by total desc""").show()

Calculating loan scores

In [4]:
credit_score_tiers = {
    "unacceptable": 0,
    "very_bad": 100,
    "bad": 250,
    "good": 500,
    "very_good": 650,
    "excellent": 800
}

In [5]:
#Reading bad data for excluding them later on
bad_customer_data_final_df = spark.read \
.format("csv") \
.option("header", True) \
.option("inferSchema", True) \
.load("../data/bad/bad_customer_data_final.csv")


In [6]:
bad_customer_data_final_df.createOrReplaceTempView("bad_data_customer")

In [7]:
ph_df = spark.sql(f"""
SELECT
    c.member_id,
    CASE
        WHEN p.last_payment_amount >= (c.monthly_installment * 0.5)
             AND p.last_payment_amount < c.monthly_installment
            THEN {credit_score_tiers["very_bad"]}
        WHEN p.last_payment_amount = c.monthly_installment
            THEN {credit_score_tiers["good"]}
        WHEN p.last_payment_amount > c.monthly_installment
             AND p.last_payment_amount <= (c.monthly_installment * 1.5)
            THEN {credit_score_tiers["very_good"]}
        WHEN p.last_payment_amount > (c.monthly_installment * 1.5)
            THEN {credit_score_tiers["excellent"]}
        ELSE {credit_score_tiers["unacceptable"]}
    END AS last_payment_pts,

    CASE
        WHEN p.total_payment_received >= (c.funded_amount * 0.5)
            THEN {credit_score_tiers["very_good"]}
        WHEN p.total_payment_received < (c.funded_amount * 0.5)
             AND p.total_payment_received > 0
            THEN {credit_score_tiers["good"]}
        WHEN p.total_payment_received = 0
             OR p.total_payment_received IS NULL
            THEN {credit_score_tiers["unacceptable"]}
    END AS total_payment_pts

FROM lending_club.loans_repayments p
JOIN lending_club.loans c
    ON c.loan_id = p.loan_id
WHERE c.member_id NOT IN (
    SELECT member_id FROM bad_data_customer
)
""")

26/02/04 10:43:33 WARN ObjectStore: Version information not found in metastore. hive.metastore.schema.verification is not enabled so recording the schema version 2.3.0
26/02/04 10:43:33 WARN ObjectStore: setMetaStoreSchemaVersion called but recording version is disabled: version = 2.3.0, comment = Set by MetaStore himanshuhegde@127.0.0.1


In [8]:
ph_df.createOrReplaceTempView("ph_pts")

In [ ]:
spark.sql("select * from ph_pts").show()

In [ ]:
# calculation criterion 2 - loan defaulters history ldh

In [9]:
ldh_ph_df = spark.sql(f"""
    select p.*,
    CASE 
    WHEN d.delinq_2yrs = 0 
        THEN {credit_score_tiers["excellent"]} 
    WHEN d.delinq_2yrs BETWEEN 1 AND 2 
        THEN {credit_score_tiers["bad"]} 
    WHEN d.delinq_2yrs BETWEEN 3 AND 5 
        THEN {credit_score_tiers["very_bad"]} 
    WHEN d.delinq_2yrs > 5 OR d.delinq_2yrs IS NULL 
        THEN {credit_score_tiers["unacceptable"]} 
    END AS delinq_pts, 
    CASE 
    WHEN l.pub_rec = 0 
        THEN {credit_score_tiers["excellent"]} 
    WHEN l.pub_rec BETWEEN 1 AND 2 
        THEN {credit_score_tiers["bad"]} 
    WHEN l.pub_rec BETWEEN 3 AND 5 
        THEN {credit_score_tiers["very_bad"]} 
    WHEN l.pub_rec > 5 OR l.pub_rec IS NULL 
        THEN {credit_score_tiers["very_bad"]} 
    END AS public_records_pts, 
    CASE 
    WHEN l.pub_rec_bankruptcies = 0 
        THEN {credit_score_tiers["excellent"]} 
    WHEN l.pub_rec_bankruptcies BETWEEN 1 AND 2 
        THEN {credit_score_tiers["bad"]} 
    WHEN l.pub_rec_bankruptcies BETWEEN 3 AND 5 
        THEN {credit_score_tiers["very_bad"]} 
    WHEN l.pub_rec_bankruptcies > 5 OR l.pub_rec_bankruptcies IS NULL 
        THEN {credit_score_tiers["very_bad"]}
    END as public_bankruptcies_pts, 
    CASE 
    WHEN l.inq_last_6mths = 0 
        THEN {credit_score_tiers["excellent"]} 
    WHEN l.inq_last_6mths BETWEEN 1 AND 2 
        THEN {credit_score_tiers["bad"]} 
    WHEN l.inq_last_6mths BETWEEN 3 AND 5 
        THEN {credit_score_tiers["very_bad"]} 
    WHEN l.inq_last_6mths > 5 OR l.inq_last_6mths IS NULL 
        THEN {credit_score_tiers["unacceptable"]} 
    END AS enq_pts 
    FROM lending_club.loans_defaulters_detail_rec_enq_new l 
    INNER JOIN lending_club.loans_defaulters_delinq_new d ON d.member_id = l.member_id  
    INNER JOIN ph_pts p ON p.member_id = l.member_id where l.member_id NOT IN (select member_id from bad_data_customer)
    """)

In [10]:
ldh_ph_df.createOrReplaceTempView("ldh_ph_pts")

In [ ]:
spark.sql("select * from ldh_ph_pts").show()

In [11]:
fh_ldh_ph_df = spark.sql(f"""select ldef.*, 
    CASE 
    WHEN LOWER(l.loan_status) LIKE '%fully paid%'
        THEN {credit_score_tiers["excellent"]} 
    WHEN LOWER(l.loan_status) LIKE '%current%' 
        THEN {credit_score_tiers["good"]} 
    WHEN LOWER(l.loan_status) LIKE '%in grace period%' 
        THEN {credit_score_tiers["bad"]} 
    WHEN LOWER(l.loan_status) LIKE '%late (16-30 days)%' OR LOWER(l.loan_status) LIKE '%late (31-120 days)%' 
        THEN {credit_score_tiers["very_bad"]} 
    WHEN LOWER(l.loan_status) LIKE '%charged off%' 
        THEN {credit_score_tiers["unacceptable"]}
    else {credit_score_tiers["unacceptable"]} 
    END AS loan_status_pts, 
    CASE 
    WHEN LOWER(a.home_ownership) LIKE '%own' 
        THEN {credit_score_tiers["excellent"]} 
    WHEN LOWER(a.home_ownership) LIKE '%rent' 
        THEN {credit_score_tiers["good"]} 
    WHEN LOWER(a.home_ownership) LIKE '%mortgage' 
        THEN {credit_score_tiers["bad"]} 
    WHEN LOWER(a.home_ownership) LIKE '%any' OR LOWER(a.home_ownership) IS NULL 
        THEN {credit_score_tiers["very_bad"]} 
    END AS home_pts, 
    CASE 
    WHEN l.funded_amount <= (a.total_high_credit_limit * 0.10) 
        THEN {credit_score_tiers["excellent"]} 
    WHEN l.funded_amount > (a.total_high_credit_limit * 0.10) AND l.funded_amount <= (a.total_high_credit_limit * 0.20) 
        THEN {credit_score_tiers["very_good"]} 
    WHEN l.funded_amount > (a.total_high_credit_limit * 0.20) AND l.funded_amount <= (a.total_high_credit_limit * 0.30) 
        THEN {credit_score_tiers["good"]} 
    WHEN l.funded_amount > (a.total_high_credit_limit * 0.30) AND l.funded_amount <= (a.total_high_credit_limit * 0.50) 
        THEN {credit_score_tiers["bad"]} 
    WHEN l.funded_amount > (a.total_high_credit_limit * 0.50) AND l.funded_amount <= (a.total_high_credit_limit * 0.70) 
        THEN {credit_score_tiers["very_bad"]} 
    WHEN l.funded_amount > (a.total_high_credit_limit * 0.70) 
        THEN {credit_score_tiers["unacceptable"]}
    else {credit_score_tiers["unacceptable"]} 
    END AS credit_limit_pts, 
    CASE 
    WHEN (a.grade) = 'A' and (a.sub_grade)='A1' 
        THEN {credit_score_tiers["excellent"]} 
    WHEN (a.grade) = 'A' and (a.sub_grade)='A2' 
        THEN ({credit_score_tiers["excellent"]} * 0.95) 
    WHEN (a.grade) = 'A' and (a.sub_grade)='A3' 
        THEN ({credit_score_tiers["excellent"]} * 0.90) 
    WHEN (a.grade) = 'A' and (a.sub_grade)='A4' 
        THEN ({credit_score_tiers["excellent"]} * 0.85) 
    WHEN (a.grade) = 'A' and (a.sub_grade)='A5' 
        THEN ({credit_score_tiers["excellent"]} * 0.80) 
    WHEN (a.grade) = 'B' and (a.sub_grade)='B1' 
        THEN {credit_score_tiers["very_good"]} 
    WHEN (a.grade) = 'B' and (a.sub_grade)='B2' 
        THEN ({credit_score_tiers["very_good"]} * 0.95) 
    WHEN (a.grade) = 'B' and (a.sub_grade)='B3' 
        THEN ({credit_score_tiers["very_good"]} * 0.90) 
    WHEN (a.grade) = 'B' and (a.sub_grade)='B4' 
        THEN ({credit_score_tiers["very_good"]} * 0.85) 
    WHEN (a.grade) = 'B' and (a.sub_grade)='B5' 
        THEN ({credit_score_tiers["very_good"]} * 0.80) 
    WHEN (a.grade) = 'C' and (a.sub_grade)='C1' 
        THEN {credit_score_tiers["good"]} 
    WHEN (a.grade) = 'C' and (a.sub_grade)='C2' 
        THEN ({credit_score_tiers["good"]} * 0.95) 
    WHEN (a.grade) = 'C' and (a.sub_grade)='C3' 
        THEN ({credit_score_tiers["good"]} * 0.90) 
    WHEN (a.grade) = 'C' and (a.sub_grade)='C4' 
        THEN ({credit_score_tiers["good"]} * 0.85) 
    WHEN (a.grade) = 'C' and (a.sub_grade)='C5' 
        THEN ({credit_score_tiers["good"]} * 0.80) 
    WHEN (a.grade) = 'D' and (a.sub_grade)='D1' THEN ({credit_score_tiers["bad"]}) 
    WHEN (a.grade) = 'D' and (a.sub_grade)='D2' THEN ({credit_score_tiers["bad"]} * 0.95) 
    WHEN (a.grade) = 'D' and (a.sub_grade)='D3' THEN ({credit_score_tiers["bad"]} * 0.90) 
    WHEN (a.grade) = 'D' and (a.sub_grade)='D4' THEN ({credit_score_tiers["bad"]} * 0.85) 
    WHEN (a.grade) = 'D' and (a.sub_grade)='D5' THEN ({credit_score_tiers["bad"]} * 0.80) 
    WHEN (a.grade) = 'E' and (a.sub_grade)='E1' THEN {credit_score_tiers["very_bad"]} 
    WHEN (a.grade) = 'E' and (a.sub_grade)='E2' THEN ({credit_score_tiers["very_bad"]} * 0.95) 
    WHEN (a.grade) = 'E' and (a.sub_grade)='E3' THEN ({credit_score_tiers["very_bad"]} * 0.90) 
    WHEN (a.grade) = 'E' and (a.sub_grade)='E4' THEN ({credit_score_tiers["very_bad"]} * 0.85) 
    WHEN (a.grade) = 'E' and (a.sub_grade)='E5' THEN ({credit_score_tiers["very_bad"]} * 0.80) 
    WHEN (a.grade) in ('F', 'G') THEN {credit_score_tiers["unacceptable"]} 
    END AS grade_pts 
    FROM ldh_ph_pts ldef 
    INNER JOIN lending_club.loans l ON ldef.member_id = l.member_id 
    INNER JOIN lending_club.customers_new a ON a.member_id = ldef.member_id where ldef.member_id NOT IN (select member_id from bad_data_customer)
   """) 

In [12]:
fh_ldh_ph_df.createOrReplaceTempView("fh_ldh_ph_pts")

In [14]:
spark.sql("select * from fh_ldh_ph_pts").show()

+--------------------+----------------+-----------------+----------+------------------+-----------------------+-------+---------------+--------+----------------+---------+
|           member_id|last_payment_pts|total_payment_pts|delinq_pts|public_records_pts|public_bankruptcies_pts|enq_pts|loan_status_pts|home_pts|credit_limit_pts|grade_pts|
+--------------------+----------------+-----------------+----------+------------------+-----------------------+-------+---------------+--------+----------------+---------+
|0000036e9afe019a6...|             800|              650|       250|               800|                    800|    100|            800|     250|             800|   500.00|
|000152208b3e77b5b...|             500|              650|       800|               800|                    800|    800|            500|     500|             650|   250.00|
|000170b4ccb292792...|             800|              650|       250|               800|                    800|    250|            800|     

In [16]:
# Final Credit Score Calculation
# 1. Payment History = 20%
# 2. Loan Defaults = 45%
# 3. Financial Health = 35%

In [18]:
credit_score = spark.sql("""SELECT member_id, 
    ((last_payment_pts+total_payment_pts)*0.20) as payment_history_pts, 
    ((delinq_pts + public_records_pts + public_bankruptcies_pts + enq_pts) * 0.45) as defaulters_history_pts, 
    ((loan_status_pts + home_pts + credit_limit_pts + grade_pts)*0.35) as financial_health_pts 
    FROM fh_ldh_ph_pts""")

In [19]:
credit_score.show()

+--------------------+-------------------+----------------------+--------------------+
|           member_id|payment_history_pts|defaulters_history_pts|financial_health_pts|
+--------------------+-------------------+----------------------+--------------------+
|0000036e9afe019a6...|             290.00|                877.50|            822.5000|
|000152208b3e77b5b...|             230.00|               1440.00|            665.0000|
|000170b4ccb292792...|             290.00|                945.00|            852.2500|
|0001cfa200f7480b9...|             230.00|                450.00|            665.0000|
|00024adf1230710bd...|             200.00|               1192.50|            682.5000|
|00026136ec721b938...|             200.00|                945.00|            653.6250|
|00030e831c078f92a...|             230.00|               1440.00|            595.8750|
|0004656412a18b9c1...|             290.00|               1192.50|            988.7500|
|0004f290201c29d93...|             200.00| 

In [20]:
final_credit_score = credit_score.withColumn('credit_score', credit_score.payment_history_pts + credit_score.defaulters_history_pts + credit_score.financial_health_pts)

In [21]:
final_credit_score.createOrReplaceTempView("credit_score_eval")

In [22]:
final_credit_score.show()

+--------------------+-------------------+----------------------+--------------------+------------+
|           member_id|payment_history_pts|defaulters_history_pts|financial_health_pts|credit_score|
+--------------------+-------------------+----------------------+--------------------+------------+
|0000036e9afe019a6...|             290.00|                877.50|            822.5000|   1990.0000|
|000152208b3e77b5b...|             230.00|               1440.00|            665.0000|   2335.0000|
|000170b4ccb292792...|             290.00|                945.00|            852.2500|   2087.2500|
|0001cfa200f7480b9...|             230.00|                450.00|            665.0000|   1345.0000|
|00024adf1230710bd...|             200.00|               1192.50|            682.5000|   2075.0000|
|00026136ec721b938...|             200.00|                945.00|            653.6250|   1798.6250|
|00030e831c078f92a...|             230.00|               1440.00|            595.8750|   2265.8750|


In [23]:
final_credit_score.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "../data/processed/credit_score") \
.save()